In [1]:
# # _회귀분석 이런 저런 방법들. 
# 일단 SLM 회귀분석 방법 및 추정량의 불편성, 일치성, 모의실험으로 보이기.

import os
import numpy as np
import pandas as pd 
import matplotlib.pyplot as plt
import statsmodels.api as sm
# scikit-learn 모듈. 머신러닝 모듈임.
from sklearn.linear_model import LinearRegression


자료는 다음 모형에 따라 생성됨. 
\begin{align}
y_i &= \beta_0 + \beta_1 x_i + e_i 
\end{align}
where  
$ e_i \sim iidN(0,\sigma^2) $,  $ i=1,2,\dots,n $.

In [2]:
# 단순 회귀 실험용 자료 생성, 모의실험 
# 엄밀하게는 주어진 X, 이것은 확률변수 아님. 고로 평균함수도 마찬가지.
# 종속변수는 평균함수에 임의 요소 추가. 
# seed = 135678942 
# rng = np.random.default_rng(seed)  # 난수생성 준비. 객체. 크기나 종류, 이런 것 미정.
#                                        # 이 상대는 계속 동일한 자료를 생성. 루프 밖으로 내는 아이디어? 
def rngd_slm_y(beta_0, beta_1, x_dat, slm_std) :
    n_size = len(x_dat)       # x는 non-random.
    # 난수 생성: 
    slm_e = rng.normal(loc = 0, scale= slm_std, size=n_size)
    y = beta_0 + beta_1 * x_dat + slm_e 
    return y 



### 데이터 생성, 회귀분석 준비

In [3]:
# 데이터 파일 읽기. github.com
dat_url = 'https://github.com/bahn28/gamja/blob/main/cs_nns_gndr_hgt.csv?raw=true'
df_dat = pd.read_csv(dat_url) 
df_dat.head() 

n_size = 210 
x_dat = df_dat['ht'].head(n_size)   # use real data for regressors X in simulation.
beta_0 = 30 
beta_1 = 0.5
slm_std = 15 
seed = 135678942


# X = x_dat.to_numpy()  # 2차원 배열(행렬: 샘플수, 특성수), 그런데 시리즈는 1차원 배열임.
X = x_dat.to_numpy().reshape(-1,1) 
            # 대안1. numpy 후 reshape 적용. 1차원 배열 (N,) --> 2차원 배열 (N, 1)로 변환
            # 대안2: 시리즈에서 읽을 때 [] 두번 감싸고 넘파이. x_dat = df_dat[['ht]].to_numpy()  

rng = np.random.default_rng(seed)  # 난수생성 준비. 객체. 크기나 종류, 이런 것 미정.
                                       # 이 상대는 계속 동일한 자료를 생성. 루프 밖으로 내는 아이디어? 
                                       # 아래 함수 내에 난수 생성 들어감.

# 여기서 부터 반복 예정, 실험이니까, 다음 셀. 
y = rngd_slm_y(beta_0, beta_1, x_dat, slm_std) 

    # plt.scatter(x_dat, y)
    # works 
    # type(x_dat)
    # type(y)
    # x_dat, y는 pandas series 
        
y = y.to_numpy()     

print("기울기의 표준오차(이론)")
seobeta1 = slm_std / ( ( x_dat**2).sum() - len(x_dat)*( x_dat.mean() )**2 )**(0.5) 
print(seobeta1)  

기울기의 표준오차(이론)
0.12735999012978222


## 최소제곱추정, 단순모형.

In [4]:
# statsmodels 모듈 이용한 회귀분석 결과
# X에 상수항 추가.
X_con = sm.add_constant(X)
model = sm.OLS(y, X_con).fit()
print(model.summary())

                            OLS Regression Results                            
Dep. Variable:                      y   R-squared:                       0.124
Model:                            OLS   Adj. R-squared:                  0.120
Method:                 Least Squares   F-statistic:                     29.51
Date:                Sun, 23 Aug 2026   Prob (F-statistic):           1.54e-07
Time:                        03:48:38   Log-Likelihood:                -866.01
No. Observations:                 210   AIC:                             1736.
Df Residuals:                     208   BIC:                             1743.
Df Model:                           1                                         
Covariance Type:            nonrobust                                         
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const         -2.8660     20.580     -0.139      0.8

In [5]:
     
model = LinearRegression()  
model.fit(X,y)
beta_1_hat = model.coef_       # 이건 array
beta_0_hat = model.intercept_  # 이건 float 
    # print(model.coef_)       # 기울기
    # print(model.intercept_)  # 절편, 모형에 상수항 자동 추가네.
    # print(beta_0_hat, beta_1_hat) 
    # type(beta_0_hat)  # 이건  numpy.float 
    # type(beta_1_hat)  # 이건 numpy.ndarray
beta = ((model.intercept_, model.coef_[0]))  # 리스트로 묶어. 
beta_hat = np.array(beta)  # ndarray로 변환.

print("단 1회 실행 결과 ")
print(f"표본크기 {n_size}, 참 절편 {beta_0}, 참 기울기 {beta_1} ")
print(f" 추정치 절편: {beta_0_hat}, 기울기: {beta_1_hat} ")
print(beta)      # 이건 1행으로 짝 맞춰 계속 나열, 즉 리스트 형태. 1회 실행이라 1행으로 2개 나열/리스트
print(beta_hat)  # 이건 행마다 짝을 맞추고, 행을 아래로 나열. 즉 행렬 형태. 1회 실행이라 1행, 2열. 




단 1회 실행 결과 
표본크기 210, 참 절편 30, 참 기울기 0.5 
 추정치 절편: -2.8659720075003605, 기울기: [0.69304299] 
(np.float64(-2.8659720075003605), np.float64(0.6930429925366746))
[-2.86597201  0.69304299]


## 회귀분석 모의실험
여러번 반복하여 회귀계수 추정치가 어떻게 분포하는지 보자.

평균적으로 어떤 값을 갖는지, 표준오차의 크기는 얼마나 되는지. 

$ X $는 확률변수가 아님. 여기서는 real data를 사용하여 고정시킴.

$ y $는 SLM 적용. 즉, 오차를 발생시키고, 평균함수에 더하여 종속변수 생성함.

따라서 $ \beta_0, \beta_1 $은 일종의 '하늘의 뜻'임. '땅에서는 모름.'

In [6]:
# 1. 모의실험. 
#    위 셀의 OLS를 여러번 반복하는 것임. 
##   반복 횟수, 표본 크기 등 설정. 
n_iter = 1342
n_size = 4321 
x_dat = df_dat['ht'].head(n_size)
beta_0 = 30 
beta_1 = 0.5
slm_std = 15 
seed = 135678942


# X = x_dat.to_numpy()  # 2차원 배열(행렬: 샘플수, 특성수), 그런데 시리즈는 1차원 배열임.
X = x_dat.to_numpy().reshape(-1,1) 
            # 대안1. numpy 후 reshape 적용. 1차원 배열 (N,) --> 2차원 배열 (N, 1)로 변환
            # 대안2: 시리즈에서 읽을 때 [] 두번 감싸고 넘파이. x_dat = df_dat[['ht]].to_numpy()  

rng = np.random.default_rng(seed)  # 난수생성 준비. 객체. 크기나 종류, 이런 것 미정.
                                       # 이 상대는 계속 동일한 자료를 생성. 루프 밖으로 내는 아이디어? 
                                       # 아래 함수 내에 난수 생성 들어감.

beta_hat = []   # 반복 결과를 list로 받자. 나중에 ndarray로 전환.
for i in range( n_iter) :  # range(0, n_iter, 1) 과 동일함....
# 여기서 부터 반복 예정, 실험이니까. 
    y = rngd_slm_y(beta_0, beta_1, x_dat, slm_std) 
    y = y.to_numpy()          
    model = LinearRegression()  
    model.fit(X,y)
    beta_1_hat = model.coef_       # 이건 array
    beta_0_hat = model.intercept_  # 이건 float 
    # print("몇 번째 반복인가? ", i, ", 기울기:",  beta_1_hat)  # 반복하면서 추정한다는 걸 보여주자. 

    beta_hat.append((model.intercept_, model.coef_[0]))  # 리스트로 묶어서 추가한다고. 
    # 최종 결과, beta. 리스트임. 

beta_hat = np.array(beta_hat)  # ndarray로 변환.
# print(beta)      # 이건 1행으로 짝 맞춰 계속 나열, 즉 리스트 형태.
# print(beta_hat)  # 이건 행마다 짝을 맞추고, 행을 아래로 나열. 즉 행렬 형태.  

mean_b_0 = np.mean( beta_hat[:,0] ) # 인덱스 0인 array 평균. 
mean_b_1 = np.mean( beta_hat[:,1] ) # 인덱스 1인 array 평균. 
se_b_1 = np.std(beta_hat[:,1], ddof=1)  

print(f"반복횟수 {n_iter}, 표본크기 {n_size}, 모델 설정 값: 절편 {beta_0}, 기울기 {beta_1} ")
print(f"반복 추정  절편의 평균: {mean_b_0}, 이 값이 참 절편(모수) {beta_0}에 가까운가?" ) #, mean_b_0)
print(f"반복 추정 기울기의 평균: {mean_b_1}, 이 값이 참 기울기(모수) {beta_1}에 가까운가?" ) #, mean_b_1)
print(f"반복 추정 기울기의 표준오차: {se_b_1}  ")  
print(f"표본 크기({n_size})가 증가하면, 반복 추정 기울기의 표준오차는 어떻게 될까?") 
print("표본크기, 반복횟수 등을 조정하여 살펴볼 수 있음. ")

print("기울기의 표준오차(이론)")
seobeta1 = slm_std / ( ( x_dat**2).sum() - len(x_dat)*( x_dat.mean() )**2 )**(0.5) 
print(seobeta1)  

반복횟수 1342, 표본크기 4321, 모델 설정 값: 절편 30, 기울기 0.5 
반복 추정  절편의 평균: 29.890143142262286, 이 값이 참 절편(모수) 30에 가까운가?
반복 추정 기울기의 평균: 0.5007165198574132, 이 값이 참 기울기(모수) 0.5에 가까운가?
반복 추정 기울기의 표준오차: 0.026895781203112028  
표본 크기(4321)가 증가하면, 반복 추정 기울기의 표준오차는 어떻게 될까?
표본크기, 반복횟수 등을 조정하여 살펴볼 수 있음. 
기울기의 표준오차(이론)
0.026868784700614597


\begin{align}
\hat \beta & = (X'X)^{-1}X'y \\ 
E(\hat\beta) & = \beta \\
V(\hat\beta) & = \sigma^2 (X'X)^{-1} 
\end{align}